# NEXORA CARE FLOW
## Appointment Records – Feature Engineering

### Project Overview
This feature engineering stage prepares the Nexora Care Flow appointment records for time-series forecasting by transforming the cleaned data into a structured clinic-level dataset.

The analysis aggregates appointment records into daily clinic volumes and creates features that capture calendar patterns, holiday and seasonal effects, previous appointment demand, and recent demand trends. These features will provide the foundation for developing a forecasting model to predict future appointment volumes.

### Feature Engineering Objectives

- Aggregate appointment records into daily appointment volumes at clinic level.

- Create calendar features including day of week, month, quarter, year, and weekend indicators.

- Incorporate holiday and seasonal information into the clinic-level dataset.

- Create historical demand features using previous appointment volumes.

- Calculate rolling averages to capture recent demand trends.

- Prepare a structured dataset for the subsequent forecasting and model validation stage.


In [6]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [33]:
# Load the cleaned eda dataset
df = pd.read_csv(r'C:\Users\telvi\Downloads\AMDARI\Nexora_Care_Flow\data\processed_data\AppointmentRecords_eda.csv'
)

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           129353 non-null  object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
 15  DayOfWeek        

In [9]:
# CSV files do not preserve pandas datetime types, convert AppointmentDateTime again:
df['AppointmentDateTime'] = pd.to_datetime(
    df['AppointmentDateTime']
)

In [10]:
# confirm if the AppointmentDateTime has change to datetime
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   AppointmentID         129353 non-null  int64         
 1   ClinicID              129353 non-null  int64         
 2   ClinicName            129353 non-null  object        
 3   ClinicType            129353 non-null  object        
 4   PatientID             129353 non-null  int64         
 5   ProviderID            129353 non-null  int64         
 6   AppointmentDateTime   129353 non-null  datetime64[ns]
 7   AppointmentType       129353 non-null  object        
 8   Status                129353 non-null  object        
 9   BookingChannel        129353 non-null  object        
 10  BookingLeadTimeDays   129353 non-null  int64         
 11  IsHoliday             129353 non-null  bool          
 12  HolidayName           129353 non-null  object        
 13 

In [11]:
# 1. Create clinic-level daily appointment volume
# Aggregate appointment records to daily volume for each clinic.
# This creates the main time-series dataset for forecasting.
# This will changes the data from one row per appointment to one row per clinic per day, 
# which is much more appropriate for appointment-volume forecasting.

clinic_daily = (
    df.groupby(
        [
            'ClinicID',
            'ClinicName',
            pd.Grouper(
                key='AppointmentDateTime',
                freq='D'
            )
        ]
    )
    .size()
    .reset_index(
        name='AppointmentCount'
    )
)

clinic_daily.head()

,ClinicID,ClinicName,AppointmentDateTime,AppointmentCount
0,1,Nexora Care - Riverside,2024-01-01,13
1,1,Nexora Care - Riverside,2024-01-02,75
2,1,Nexora Care - Riverside,2024-01-03,66
3,1,Nexora Care - Riverside,2024-01-04,62
4,1,Nexora Care - Riverside,2024-01-05,36


In [12]:
# 2. Rename the date column
# Rename AppointmentDateTime to Date because the data
# has now been aggregated to daily appointment volume.

clinic_daily = clinic_daily.rename(
    columns={
        'AppointmentDateTime': 'Date'
    }
)

clinic_daily.head()

,ClinicID,ClinicName,Date,AppointmentCount
0,1,Nexora Care - Riverside,2024-01-01,13
1,1,Nexora Care - Riverside,2024-01-02,75
2,1,Nexora Care - Riverside,2024-01-03,66
3,1,Nexora Care - Riverside,2024-01-04,62
4,1,Nexora Care - Riverside,2024-01-05,36


In [13]:
# 3. Create calendar features
# Create calendar-based features from the appointment date.
# These features capture recurring patterns such as
# day-of-week, month, quarter, and weekend effects.

clinic_daily['DayOfWeek'] = (
    clinic_daily['Date'].dt.dayofweek
)

clinic_daily['DayName'] = (
    clinic_daily['Date'].dt.day_name()
)

clinic_daily['Day'] = (
    clinic_daily['Date'].dt.day
)

clinic_daily['Month'] = (
    clinic_daily['Date'].dt.month
)

clinic_daily['MonthName'] = (
    clinic_daily['Date'].dt.month_name()
)

clinic_daily['Quarter'] = (
    clinic_daily['Date'].dt.quarter
)

clinic_daily['Year'] = (
    clinic_daily['Date'].dt.year
)

clinic_daily['WeekOfYear'] = (
    clinic_daily['Date']
    .dt.isocalendar()
    .week
    .astype(int)
)

clinic_daily['IsWeekend'] = (
    clinic_daily['DayOfWeek'] >= 5
).astype(int)

In [14]:
# Check the result:
# Display the newly created calendar features
clinic_daily[
    [
        'ClinicID',
        'ClinicName',
        'Date',
        'AppointmentCount',
        'DayOfWeek',
        'DayName',
        'Month',
        'MonthName',
        'Quarter',
        'Year',
        'WeekOfYear',
        'IsWeekend'
    ]
].head(10)

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,DayName,Month,MonthName,Quarter,Year,WeekOfYear,IsWeekend
0,1,Nexora Care - Riverside,2024-01-01,13,0,Monday,1,January,1,2024,1,0
1,1,Nexora Care - Riverside,2024-01-02,75,1,Tuesday,1,January,1,2024,1,0
2,1,Nexora Care - Riverside,2024-01-03,66,2,Wednesday,1,January,1,2024,1,0
3,1,Nexora Care - Riverside,2024-01-04,62,3,Thursday,1,January,1,2024,1,0
4,1,Nexora Care - Riverside,2024-01-05,36,4,Friday,1,January,1,2024,1,0
5,1,Nexora Care - Riverside,2024-01-06,17,5,Saturday,1,January,1,2024,1,1
6,1,Nexora Care - Riverside,2024-01-08,55,0,Monday,1,January,1,2024,2,0
7,1,Nexora Care - Riverside,2024-01-09,52,1,Tuesday,1,January,1,2024,2,0
8,1,Nexora Care - Riverside,2024-01-10,72,2,Wednesday,1,January,1,2024,2,0
9,1,Nexora Care - Riverside,2024-01-11,71,3,Thursday,1,January,1,2024,2,0


In [15]:
# 4. Add holiday information
# Note Your original dataset already contains IsHoliday and HolidayName, so we can retain this 
# information at the daily clinic level.

# Create a daily holiday lookup from the original appointment data.
# This allows holiday information to be added to the
# clinic-level time-series dataset.

holiday_lookup = (
    df[
        [
            'AppointmentDateTime',
            'IsHoliday',
            'HolidayName'
        ]
    ]
    .assign(
        Date=lambda x:
        x['AppointmentDateTime'].dt.normalize()
    )
    [
        [
            'Date',
            'IsHoliday',
            'HolidayName'
        ]
    ]
    .drop_duplicates(
        subset=['Date']
    )
)

holiday_lookup.head()

,Date,IsHoliday,HolidayName
0,2024-01-01,True,New Year's Day
31,2024-01-02,False,No Holiday
252,2024-01-03,False,No Holiday
419,2024-01-04,False,No Holiday
590,2024-01-05,False,No Holiday


In [16]:
# Lets Merge it into the clinic-level data:
# Add holiday indicators to the clinic-level dataset.

clinic_daily = clinic_daily.merge(
    holiday_lookup,
    on='Date',
    how='left'
)

clinic_daily.head()

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,DayName,Day,Month,MonthName,Quarter,Year,WeekOfYear,IsWeekend,IsHoliday,HolidayName
0,1,Nexora Care - Riverside,2024-01-01,13,0,Monday,1,1,January,1,2024,1,0,True,New Year's Day
1,1,Nexora Care - Riverside,2024-01-02,75,1,Tuesday,2,1,January,1,2024,1,0,False,No Holiday
2,1,Nexora Care - Riverside,2024-01-03,66,2,Wednesday,3,1,January,1,2024,1,0,False,No Holiday
3,1,Nexora Care - Riverside,2024-01-04,62,3,Thursday,4,1,January,1,2024,1,0,False,No Holiday
4,1,Nexora Care - Riverside,2024-01-05,36,4,Friday,5,1,January,1,2024,1,0,False,No Holiday


In [17]:
# 5. lets Add seasonal information
# our dataset already contains SeasonFlag, with categories such as Standard, Flu Season, and Allergy Season.

# Create a daily seasonal lookup from the original dataset.
# SeasonFlag captures standard, flu-season, and allergy-season periods.

season_lookup = (
    df[
        [
            'AppointmentDateTime',
            'SeasonFlag'
        ]
    ]
    .assign(
        Date=lambda x:
        x['AppointmentDateTime'].dt.normalize()
    )
    [
        [
            'Date',
            'SeasonFlag'
        ]
    ]
    .drop_duplicates(
        subset=['Date']
    )
)

season_lookup.head()

,Date,SeasonFlag
0,2024-01-01,Flu Season
31,2024-01-02,Flu Season
252,2024-01-03,Flu Season
419,2024-01-04,Flu Season
590,2024-01-05,Flu Season


In [18]:
# Merge the seasonal information:
# Add the seasonal classification to the clinic-level dataset.

clinic_daily = clinic_daily.merge(
    season_lookup,
    on='Date',
    how='left'
)

clinic_daily.head()

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,DayName,Day,Month,MonthName,Quarter,Year,WeekOfYear,IsWeekend,IsHoliday,HolidayName,SeasonFlag
0,1,Nexora Care - Riverside,2024-01-01,13,0,Monday,1,1,January,1,2024,1,0,True,New Year's Day,Flu Season
1,1,Nexora Care - Riverside,2024-01-02,75,1,Tuesday,2,1,January,1,2024,1,0,False,No Holiday,Flu Season
2,1,Nexora Care - Riverside,2024-01-03,66,2,Wednesday,3,1,January,1,2024,1,0,False,No Holiday,Flu Season
3,1,Nexora Care - Riverside,2024-01-04,62,3,Thursday,4,1,January,1,2024,1,0,False,No Holiday,Flu Season
4,1,Nexora Care - Riverside,2024-01-05,36,4,Friday,5,1,January,1,2024,1,0,False,No Holiday,Flu Season


In [19]:
# 6. Sort the time series
# This step is very important before creating lag features.

# Sort the data chronologically within each clinic.
# This ensures that lag and rolling calculations use previous observations rather than future observations.

clinic_daily = (
    clinic_daily
    .sort_values(
        ['ClinicID', 'Date']
    )
    .reset_index(drop=True)
)

clinic_daily.head(10)

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,DayName,Day,Month,MonthName,Quarter,Year,WeekOfYear,IsWeekend,IsHoliday,HolidayName,SeasonFlag
0,1,Nexora Care - Riverside,2024-01-01,13,0,Monday,1,1,January,1,2024,1,0,True,New Year's Day,Flu Season
1,1,Nexora Care - Riverside,2024-01-02,75,1,Tuesday,2,1,January,1,2024,1,0,False,No Holiday,Flu Season
2,1,Nexora Care - Riverside,2024-01-03,66,2,Wednesday,3,1,January,1,2024,1,0,False,No Holiday,Flu Season
3,1,Nexora Care - Riverside,2024-01-04,62,3,Thursday,4,1,January,1,2024,1,0,False,No Holiday,Flu Season
4,1,Nexora Care - Riverside,2024-01-05,36,4,Friday,5,1,January,1,2024,1,0,False,No Holiday,Flu Season
5,1,Nexora Care - Riverside,2024-01-06,17,5,Saturday,6,1,January,1,2024,1,1,False,No Holiday,Flu Season
6,1,Nexora Care - Riverside,2024-01-08,55,0,Monday,8,1,January,1,2024,2,0,False,No Holiday,Flu Season
7,1,Nexora Care - Riverside,2024-01-09,52,1,Tuesday,9,1,January,1,2024,2,0,False,No Holiday,Flu Season
8,1,Nexora Care - Riverside,2024-01-10,72,2,Wednesday,10,1,January,1,2024,2,0,False,No Holiday,Flu Season
9,1,Nexora Care - Riverside,2024-01-11,71,3,Thursday,11,1,January,1,2024,2,0,False,No Holiday,Flu Season


In [20]:
# 7. Create lag features
# LAG FEATURES


# Previous day's appointment volume
clinic_daily['Lag_1'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .shift(1)
)

# Appointment volume from the previous week
clinic_daily['Lag_7'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .shift(7)
)

# Appointment volume from two weeks earlier
clinic_daily['Lag_14'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .shift(14)
)

# Appointment volume from four weeks earlier
clinic_daily['Lag_28'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .shift(28)
)

In [21]:
# Check the lag features:
# Check the newly created lag features

clinic_daily[
    [
        'ClinicID',
        'Date',
        'AppointmentCount',
        'Lag_1',
        'Lag_7',
        'Lag_14',
        'Lag_28'
    ]
].head(35)

,ClinicID,Date,AppointmentCount,Lag_1,Lag_7,Lag_14,Lag_28
0,1,2024-01-01,13,NaN,NaN,NaN,NaN
1,1,2024-01-02,75,13.0,NaN,NaN,NaN
2,1,2024-01-03,66,75.0,NaN,NaN,NaN
3,1,2024-01-04,62,66.0,NaN,NaN,NaN
4,1,2024-01-05,36,62.0,NaN,NaN,NaN
5,1,2024-01-06,17,36.0,NaN,NaN,NaN
6,1,2024-01-08,55,17.0,NaN,NaN,NaN
7,1,2024-01-09,52,55.0,13.0,NaN,NaN
8,1,2024-01-10,72,52.0,75.0,NaN,NaN
9,1,2024-01-11,71,72.0,66.0,NaN,NaN


In [22]:
# 8. Create rolling averages

# ROLLING AVERAGES

# Calculate the previous 7-day average appointment volume.
# shift(1) prevents the current day's value from being
# included in its own rolling average.

clinic_daily['RollingMean_7'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(window=7)
         .mean()
    )
)


# Calculate the previous 14-day average appointment volume.

clinic_daily['RollingMean_14'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(window=14)
         .mean()
    )
)


# Calculate the previous 28-day average appointment volume.

clinic_daily['RollingMean_28'] = (
    clinic_daily
    .groupby('ClinicID')['AppointmentCount']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(window=28)
         .mean()
    )
)

In [23]:
# 9. Review the feature-engineered data
# Review the complete feature-engineered dataset.

clinic_daily[
    [
        'ClinicID',
        'ClinicName',
        'Date',
        'AppointmentCount',
        'DayOfWeek',
        'IsWeekend',
        'IsHoliday',
        'HolidayName',
        'SeasonFlag',
        'Lag_1',
        'Lag_7',
        'Lag_14',
        'Lag_28',
        'RollingMean_7',
        'RollingMean_14',
        'RollingMean_28'
    ]
].head(40)

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,IsWeekend,IsHoliday,HolidayName,SeasonFlag,Lag_1,Lag_7,Lag_14,Lag_28,RollingMean_7,RollingMean_14,RollingMean_28
0,1,Nexora Care - Riverside,2024-01-01,13,0,0,True,New Year's Day,Flu Season,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Nexora Care - Riverside,2024-01-02,75,1,0,False,No Holiday,Flu Season,13.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Nexora Care - Riverside,2024-01-03,66,2,0,False,No Holiday,Flu Season,75.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Nexora Care - Riverside,2024-01-04,62,3,0,False,No Holiday,Flu Season,66.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Nexora Care - Riverside,2024-01-05,36,4,0,False,No Holiday,Flu Season,62.0,NaN,NaN,NaN,NaN,NaN,NaN
5,1,Nexora Care - Riverside,2024-01-06,17,5,1,False,No Holiday,Flu Season,36.0,NaN,NaN,NaN,NaN,NaN,NaN
6,1,Nexora Care - Riverside,2024-01-08,55,0,0,False,No Holiday,Flu Season,17.0,NaN,NaN,NaN,NaN,NaN,NaN
7,1,Nexora Care - Riverside,2024-01-09,52,1,0,False,No Holiday,Flu Season,55.0,13.0,NaN,NaN,46.285714,NaN,NaN
8,1,Nexora Care - Riverside,2024-01-10,72,2,0,False,No Holiday,Flu Season,52.0,75.0,NaN,NaN,51.857143,NaN,NaN
9,1,Nexora Care - Riverside,2024-01-11,71,3,0,False,No Holiday,Flu Season,72.0,66.0,NaN,NaN,51.428571,NaN,NaN


In [24]:
# 10. Check missing values created by lag features

# The first few records for each clinic will naturally have missing lag/rolling values because there
#  are not enough previous observations.
# Check missing values created by lag and rolling features.
# These occur naturally at the beginning of each clinic's
# time series.

clinic_daily[
    [
        'Lag_1',
        'Lag_7',
        'Lag_14',
        'Lag_28',
        'RollingMean_7',
        'RollingMean_14',
        'RollingMean_28'
    ]
].isnull().sum()

# it is advisable we don't remove these yet if you are only completing Phase 3. They will need to be handled appropriately 
# when we prepare the data for modelling.

Lag_1               4
Lag_7              28
Lag_14             56
Lag_28            112
RollingMean_7      28
RollingMean_14     56
RollingMean_28    112
dtype: int64

In [25]:
# 11. Final feature list
# Define the final feature-engineered dataset.

feature_engineered_data = clinic_daily[
    [
        'ClinicID',
        'ClinicName',
        'Date',
        'AppointmentCount',

        # Calendar features
        'DayOfWeek',
        'DayName',
        'Day',
        'Month',
        'MonthName',
        'Quarter',
        'Year',
        'WeekOfYear',
        'IsWeekend',

        # Holiday and seasonal features
        'IsHoliday',
        'HolidayName',
        'SeasonFlag',

        # Lag features
        'Lag_1',
        'Lag_7',
        'Lag_14',
        'Lag_28',

        # Rolling features
        'RollingMean_7',
        'RollingMean_14',
        'RollingMean_28'
    ]
].copy()

feature_engineered_data.head()

,ClinicID,ClinicName,Date,AppointmentCount,DayOfWeek,DayName,Day,Month,MonthName,Quarter,...,IsHoliday,HolidayName,SeasonFlag,Lag_1,Lag_7,Lag_14,Lag_28,RollingMean_7,RollingMean_14,RollingMean_28
0,1,Nexora Care - Riverside,2024-01-01,13,0,Monday,1,1,January,1,...,True,New Year's Day,Flu Season,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Nexora Care - Riverside,2024-01-02,75,1,Tuesday,2,1,January,1,...,False,No Holiday,Flu Season,13.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Nexora Care - Riverside,2024-01-03,66,2,Wednesday,3,1,January,1,...,False,No Holiday,Flu Season,75.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Nexora Care - Riverside,2024-01-04,62,3,Thursday,4,1,January,1,...,False,No Holiday,Flu Season,66.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Nexora Care - Riverside,2024-01-05,36,4,Friday,5,1,January,1,...,False,No Holiday,Flu Season,62.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save the feature-engineered dataset
# Save the completed feature-engineered dataset
# for use in the next project phase.

feature_engineered_data.to_csv(
    'Nexora_Feature_Engineered_Data.csv',
    index=False
)

print(
    "Feature engineering completed successfully."
)

print(
    "Dataset shape:",
    feature_engineered_data.shape
)

Feature engineering completed successfully.
Dataset shape: (2457, 23)


In [31]:
# save to my processed_data folder
df.to_csv(
    r'C:\Users\telvi\Downloads\AMDARI\Nexora_Care_Flow\data\processed_data\AppointmentRecords_feature_eng.csv',
    index=False)